# 🌧️ Historical Rain Probability
Analysis of historical climate data to support outdoor event decision-making.

**Data Source:** [Open-Meteo Historical API](https://open-meteo.com/en/docs/historical-weather-api)

**Technologies:** Python, Pandas, Plotly

## 1. Imports and Configuration

In [1]:
!pip install openmeteo-requests requests-cache retry-requests
import requests
import openmeteo_requests
import requests_cache
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import locale
from datetime import date, datetime
from retry_requests import retry
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import openai

HISTORICAL_YEARS = 10        # How many years of history to analyze

# Generative AI API Key
MARITACA_API_KEY = "********"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.8/761.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.4 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19




## 2. User input
Get user desired location and validate with geolocator




In [2]:
def location_validation(user_entry):
    # geolocator instance with generic user_agent
    geolocator = Nominatim(user_agent="rain_probability_jupyter_validador")

    try:
        # return focused on Brazil
        location = geolocator.geocode(user_entry, addressdetails=True, country_codes="br")

        if location and 'address' in location.raw:
            address = location.raw['address']

            # try to capture the city
            city = address.get('city') or address.get('town') or address.get('village') or address.get('municipality')
            state = address.get('state')

            if city and state:
                return {
                    "valid": True,
                    "city": city,
                    "state": state,
                    "latitude": location.latitude,
                    "longitude": location.longitude
                }

        return {"valido": False, "error": "Localização não encontrada ou incompleta."}

    except GeocoderTimedOut:
        return {"valido": False, "error": "Timeout na API. Tente novamente."}


def data_validator(user_entry):
  try:
        data_object = datetime.strptime(user_entry, "%Y-%m-%d")
        return {"valid": True, "data": data_object}
  except ValueError:
        return {"valid": False, "error": "Data inválida ou fora do padrão YYYY-MM-DD."}


location_entry = input("Digite sua cidade e estado (ex: Araras, SP): ")
location_response = location_validation(location_entry)

if location_response["valid"]:
    print(f"✓ Dado Validado com Sucesso!")
    latitude = location_response["latitude"]
    longitude = location_response["longitude"]
    local = f"{location_response['city']}, {location_response['state']}"
else:
    print(f"✗ Erro de validação do local: {location_response['error']}")

data_entry = input("Digite a data do evento (formato YYYY-MM-DD)")
data_response = data_validator(data_entry)

if data_response["valid"]:
    print(f"✓ Dado Validado com Sucesso!")
    event_date = data_response["data"]
else:
    print(f"✗ Erro de validação da data: {data_response['error']}")


Digite sua cidade e estado (ex: Araras, SP): araras
✓ Dado Validado com Sucesso!
Digite a data do evento (formato YYYY-MM-DD)2027-01-01
✓ Dado Validado com Sucesso!


## 3. Historical Data Retrieval

In [3]:
def fetch_historical_data(latitude, longitude, years=HISTORICAL_YEARS):
    """
    Fetches historical precipitation data from Open-Meteo.
    Returns a DataFrame with columns: date, precipitation_mm
    """
    current_year = date.today().year
    start_year = current_year - years
    end_year = current_year - 1  # Last complete year

    url = "https://archive-api.open-meteo.com/v1/archive"
    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": f"{start_year}-01-01",
        "end_date": f"{end_year}-12-31",
        "daily": ["precipitation_sum", "temperature_2m_mean"],
        "timezone": "America/Sao_Paulo"
    }

    print(f"Buscando dados de {start_year} a {end_year}...")
    responses = openmeteo.weather_api(url, params = params)

    response = responses[0]
    print(f"Coordenadas: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevação: {response.Elevation()} m asl")
    print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Diferença de fuso horário para GMT+0: {response.UtcOffsetSeconds()}s")

    daily = response.Daily()
    daily_precipitation_sum = daily.Variables(0).ValuesAsNumpy()
    daily_temperature = daily.Variables(1).ValuesAsNumpy()


    daily_data = {
	    "date": pd.date_range(
		    start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		    end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		    freq = pd.Timedelta(seconds = daily.Interval()),
		    inclusive = "left"
	    ).tz_convert(response.Timezone().decode())
    }

    daily_data["precipitation_sum"] = daily_precipitation_sum
    daily_data["temperature_2m_mean"] = daily_temperature

    daily_dataframe = pd.DataFrame(data = daily_data)

    print(f"✅ {len(daily_dataframe)} dias carregados com sucesso!")
    return daily_dataframe


df = fetch_historical_data(latitude, longitude)
df.head(10)

Buscando dados de 2016 a 2025...
Coordenadas: -22.319860458374023°N -47.373321533203125°E
Elevação: 633.0 m asl
Timezone: b'America/Sao_Paulo'b'GMT-3'
Diferença de fuso horário para GMT+0: -10800s
✅ 3653 dias carregados com sucesso!


,date,precipitation_sum,temperature_2m_mean
0,2016-01-01 01:00:00-02:00,12.400001,23.616501
1,2016-01-02 01:00:00-02:00,33.000000,23.499834
2,2016-01-03 01:00:00-02:00,12.099999,23.060251
3,2016-01-04 01:00:00-02:00,2.100000,23.554001
4,2016-01-05 01:00:00-02:00,0.000000,24.579000
5,2016-01-06 01:00:00-02:00,0.000000,24.731087
6,2016-01-07 01:00:00-02:00,0.100000,25.158165
7,2016-01-08 01:00:00-02:00,0.800000,25.576918
8,2016-01-09 01:00:00-02:00,9.000000,24.470665
9,2016-01-10 01:00:00-02:00,17.600000,23.624838


## 4. Data Cleaning and Preparation

#### DataFrame columns after this:

| date | precipitation_sum | temperature_2m_mean | month | day | year | month_day | rained |

In [4]:
def prepare_data(df):
    """
    Adds auxiliary columns and removes null values.
    """
    df = df.dropna(subset=["precipitation_sum"]).copy()

    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["year"] = df["date"].dt.year
    df["month_day"] = df["date"].dt.strftime("%m-%d")  # Ex: "07-15"
    df["rained"] = df["precipitation_sum"] > 2.0     # Relevant rain: above 2mm

    return df


df = prepare_data(df)

print(f"Total de dias analisados: {len(df)}")
print(f"Dias com chuva relevante (>2mm): {df['rained'].sum()}")
print(f"\nPrimeiros registros:")
df.head()

Total de dias analisados: 3653
Dias com chuva relevante (>2mm): 1034

Primeiros registros:


,date,precipitation_sum,temperature_2m_mean,month,day,year,month_day,rained
0,2016-01-01 01:00:00-02:00,12.400001,23.616501,1,1,2016,01-01,True
1,2016-01-02 01:00:00-02:00,33.000000,23.499834,1,2,2016,01-02,True
2,2016-01-03 01:00:00-02:00,12.099999,23.060251,1,3,2016,01-03,True
3,2016-01-04 01:00:00-02:00,2.100000,23.554001,1,4,2016,01-04,True
4,2016-01-05 01:00:00-02:00,0.000000,24.579000,1,5,2016,01-05,False


## 5. Event Date Analysis

In [5]:
def analyze_event_date(df, event_date_str, window_days=7):
    """
    Calculates the probability of rain for the event date.
    Uses a window of +/- N days around the date to collect more samples.
    """
    event_date = pd.to_datetime(event_date_str)

    # Generates a list of month-day strings for the window
    window_dates = [
        (event_date + pd.Timedelta(days=d)).strftime("%m-%d")
        for d in range(-window_days, window_days + 1)
    ]

    df_window = df[df["month_day"].isin(window_dates)]

    total_days = len(df_window)
    rainy_days = df_window["rained"].sum()
    probability = (rainy_days / total_days * 100) if total_days > 0 else 0
    avg_precipitation = df_window["precipitation_sum"].mean()
    max_precipitation = df_window["precipitation_sum"].max()
    avg_temperature = df_window["temperature_2m_mean"].mean()

    print(f"📅 Data do evento: {event_date.strftime('%d/%m/%Y')}")
    print(f"🔍 Janela analisada: {window_days} dias antes e depois")
    print(f"📊 Total de dias no histórico: {total_days}")
    print(f"🌧️  Dias com chuva: {rainy_days}")
    print(f"📈 Probabilidade de chuva: {probability:.1f}%")
    print(f"💧 Precipitação média: {avg_precipitation:.1f} mm")
    print(f"⚡ Precipitação máxima registrada: {max_precipitation:.1f} mm")
    print(f"🌡️ Temperatura média: {avg_temperature:.1f} °C")

    return {
        "data": event_date,
        "probability": probability,
        "rainy_days": int(rainy_days),
        "total_days": total_days,
        "avg_precipitation": avg_precipitation,
        "max_precipitation": max_precipitation,
        "avg_temperature": avg_temperature,
        "df_window": df_window
    }


result = analyze_event_date(df, event_date)

📅 Data do evento: 01/01/2027
🔍 Janela analisada: 7 dias antes e depois
📊 Total de dias no histórico: 150
🌧️  Dias com chuva: 92
📈 Probabilidade de chuva: 61.3%
💧 Precipitação média: 7.4 mm
⚡ Precipitação máxima registrada: 43.7 mm
🌡️ Temperatura média: 23.7 °C


## 6. Charts

In [6]:
# Chart 1: Rain probability by month of the year

monthly_prob = df.groupby("month").agg(
    probability=("rained", "mean"),
    avg_precipitation=("precipitation_sum", "mean")
).reset_index()
monthly_prob["probability"] *= 100

month_names = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
               "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
monthly_prob["month_name"] = monthly_prob["month"].apply(lambda x: month_names[x - 1])

# Highlights the event month
event_month = pd.to_datetime(event_date).month
monthly_prob["color"] = monthly_prob["month"].apply(
    lambda x: "#e74c3c" if x == event_month else "#4a6fa5"
)

fig1 = go.Figure(go.Bar(
    x=monthly_prob["month_name"],
    y=monthly_prob["probability"],
    marker_color=monthly_prob["color"],
    text=monthly_prob["probability"].apply(lambda x: f"{x:.0f}%"),
    textposition="outside"
))

fig1.update_layout(
    title=f"Probabilidade de Chuva por Mês — {local} (últimos {HISTORICAL_YEARS} anos)",
    xaxis_title="Mês",
    yaxis_title="Probabilidade (%)",
    yaxis_range=[0, 100],
    plot_bgcolor="white",
    showlegend=False
)
fig1.show()

In [7]:
# Chart 2: Historical precipitation in the event window (by year)

df_window = result["df_window"].copy()

prec_por_ano = df_window.groupby("year")["precipitation_sum"].sum().reset_index()

fig2 = go.Figure(go.Bar(
    x=prec_por_ano["year"].astype(str),
    y=prec_por_ano["precipitation_sum"],
    text=prec_por_ano["precipitation_sum"].apply(lambda x: f"{x:.0f} mm"),
    textposition="outside"
))

fig2.update_layout(
    title=f"Precipitação Total na Semana do Evento por Ano — {local}",
    xaxis_title="Ano",
    yaxis_title="Precipitação total (mm)",
    plot_bgcolor="white",
    showlegend=False
)
fig2.show()

In [8]:
# Chart 3: Historical temperature in the event window (by year)

df_window = result["df_window"].copy()

prec_por_ano = df_window.groupby("year")["temperature_2m_mean"].mean().reset_index()

fig3 = go.Figure(go.Bar(
    x=prec_por_ano["year"].astype(str),
    y=prec_por_ano["temperature_2m_mean"],
    text=prec_por_ano["temperature_2m_mean"].apply(lambda x: f"{x:.0f} °C"),
    textposition="outside"
))

fig3.update_layout(
    title=f"Temperatura média na Semana do Evento por Ano — {local}",
    xaxis_title="Ano",
    yaxis_title="Temperatura média (°C)",
    plot_bgcolor="white",
    showlegend=False
)
fig3.show()

## 7. Interpretive Summary with Maritaca
The LLM receives the calculated data and generates a natural language summary.

In [9]:
def generate_maritaca_summary(result, local, api_key):
    """
    Sends data to Maritaca AI and returns an interpretive summary.
    """
    client = openai.OpenAI(
      api_key= api_key,
      base_url="https://chat.maritaca.ai/api",
    )

    prompt = f"""
    Você é um assistente que analisa dados climáticos históricos para ajudar pessoas a planejar eventos ao ar livre.

    Com base nos seguintes dados históricos para {local}:
    - Data do evento: {result['data'].strftime('%d/%m/%Y')}
    - Probabilidade histórica de chuva na semana do evento: {result['probability']:.1f}%
    - Dias com chuva registrados: {result['rainy_days']} de {result['total_days']} dias analisados
    - Precipitação média diária no período: {result['avg_precipitation']:.1f} mm
    - Precipitação máxima já registrada no período: {result['max_precipitation']:.1f} mm
    - Temperatua média diaria no período: {result['avg_temperature']:.1f} °C

    Escreva um parágrafo curto e direto em português explicando o risco de chuva para o evento,
    o que os dados significam na prática e se vale a pena ter um plano B.
    Seja objetivo e útil, sem exagerar no alarmismo nem minimizar os riscos.
    """

    message = client.responses.create(
      model="sabia-4",
      input=prompt,
    )

    return message.output[0].content[0].text


summary = generate_maritaca_summary(result, local, MARITACA_API_KEY)
print("📝 Análise:")
print(summary)

📝 Análise:
O risco de chuva para um evento em Araras, São Paulo, na semana de 01/01/2027, é significativo: a probabilidade histórica de chuva é de 61,3%, ou seja, em mais de 6 de cada 10 anos há registro de chuva nesse período. Em média, chove 7,4 mm por dia, mas já foram registrados picos de até 43,7 mm, o que indica possibilidade de pancadas fortes. Como em 92 dos 150 dias analisados houve chuva, é prudente considerar um plano B — como tendas ou local coberto — para garantir o conforto e segurança dos participantes.
